# 21. Q-Learning
**Use case:** Supply Chain — Inventory reorder policy for one SKU

A regional distributor reorders a fast-moving SKU every day, and planners use gut feel or a fixed 'order 5 daily' rule. Too much stock ties up cash and shelf space, while too little loses sales and annoys retailers. The company has daily sales history, supplier lead times and cost estimates for holding, stockouts and placing orders. Success means a reorder rule that lowers total monthly cost and is at least as good as a well-tuned (s,S) policy, explained in a form planners can check.

> 💡 Think of a new shop manager keeping a notebook: for each shelf level, they jot down how each ordering choice worked out over the following weeks. After months of notes, they simply look up the shelf level and pick the choice with the best track record. Early on they try odd choices on purpose, just to learn.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Fill a 4x3 Q-table for a toy stockroom
Hand-run 8 days of Q-learning on a 4-state version of the reorder problem and print the Q-table.

| 5W + H | |
|---|---|
| **Who** | A store inventory planner learning how an RL agent thinks about reordering one SKU. |
| **What** | A Q-table with 4 stock levels (0, 5, 10, 15) and 3 order sizes (0, 5, 10), updated one day at a time. |
| **When** | Every day after demand is known, the agent books the cost and updates one cell of the table. |
| **Where** | A single stockroom with capacity 15 units; plain numpy, no RL library. |
| **Why** | To see the Q-learning update rule work cell by cell before scaling it up. |
| **How** | Epsilon-greedy picks an order, random demand arrives, reward = minus cost, then Q is nudged toward reward plus discounted best future value. |

**🧠 ML concepts:** State, action, reward, Q-table as a lookup of expected long-run value, Epsilon-greedy exploration, Bellman / temporal-difference update, Discount factor gamma  
**🐍 Python concepts:** Lists and list.index(), numpy 2-D arrays and argmax, Functions returning tuples, for loops with range, f-string alignment, enumerate

In [ ]:
# Q-Learning by hand: a toy 4-state inventory reorder problem for one SKU
import numpy as np

np.random.seed(42)
# 🐍 PY: a list holds the 4 possible stock levels (states) and 3 possible order sizes (actions)
states = [0, 5, 10, 15]
actions = [0, 5, 10]
# 🧠 ML: the Q-table stores "how good" each (state, action) pair is; it starts at all zeros
Q = np.zeros((len(states), len(actions)))
# 🧠 ML: alpha = learning rate, gamma = how much we value future rewards
alpha, gamma, epsilon = 0.5, 0.9, 0.2
HOLD_COST, STOCKOUT_COST, ORDER_COST = 1.0, 10.0, 2.0


def step(stock, order):
    # 🐍 PY: np.random.choice picks one random demand value from the list
    demand = np.random.choice([0, 5, 10])
    # 🐍 PY: min() caps stock at warehouse capacity of 15 units
    available = min(stock + order, 15)
    sold = min(available, demand)
    left = available - sold
    lost = demand - sold
    # 🧠 ML: reward is negative cost, so maximising reward = minimising cost
    reward = -(HOLD_COST * left + STOCKOUT_COST * lost + (ORDER_COST if order > 0 else 0))
    return left, reward, demand


s = 10
for day in range(1, 9):
    # 🐍 PY: states.index() converts a stock value (e.g. 10) into a row number (e.g. 2)
    si = states.index(s)
    # 🧠 ML: epsilon-greedy - mostly pick the best known action, sometimes explore a random one
    if np.random.rand() < epsilon:
        ai = np.random.randint(len(actions))
    else:
        ai = int(np.argmax(Q[si]))
    s2, r, d = step(s, actions[ai])
    s2i = states.index(s2)
    # 🧠 ML: the Q-learning update: nudge Q toward reward + discounted best future value
    target = r + gamma * Q[s2i].max()
    Q[si, ai] += alpha * (target - Q[si, ai])
    # 🐍 PY: f-strings with :>3 right-align numbers so the printout lines up
    print(f"Day {day}: stock={s:>2} order={actions[ai]:>2} demand={d:>2} -> stock={s2:>2} "
          f"reward={r:>6.1f} Q[{s},{actions[ai]}]={Q[si, ai]:>6.2f}")
    s = s2

print("\nQ-table (rows = stock level, cols = order size)")
print("stock | order0  order5 order10 | best")
# 🐍 PY: enumerate gives both the row number and the stock value
for i, st in enumerate(states):
    row = "  ".join(f"{v:6.2f}" for v in Q[i])
    print(f"{st:>5} | {row} | order {actions[int(np.argmax(Q[i]))]}")

**Try this next:**
- Set epsilon = 0 and rerun: does the agent ever try ordering 10?
- Raise STOCKOUT_COST to 30 and see how the Q-values for order 0 change.
- Run 200 days instead of 8 and check whether the best-order column stabilises.

## MEDIUM: Tabular Q-learning for a 21-state stockroom
Train a full tabular Q-learning agent over 1,500 simulated months and compare its policy to simple rules.

| 5W + H | |
|---|---|
| **Who** | A replenishment analyst who wants a data-driven reorder rule for one fast-moving SKU. |
| **What** | A 21x3 Q-table (stock 0-20, order 0/5/10) learned from Poisson demand with holding, stockout and fixed ordering costs. |
| **When** | Trained offline in a simulator; the learned policy would be reviewed before being used daily. |
| **Where** | One SKU in one warehouse with a 20-unit shelf limit, simulated in numpy. |
| **Why** | Fixed rules like 'order 5 every day' waste money; RL finds a state-dependent rule that balances holding against stockouts. |
| **How** | Epsilon-greedy exploration with decaying epsilon, Bellman updates every day, then a greedy evaluation against two baseline rules. |

**🧠 ML concepts:** Episodes and the exploration-exploitation trade-off, Epsilon decay schedule, Learning curve via average episode cost, Greedy policy extraction from Q, Policy evaluation against baselines  
**🐍 Python concepts:** numpy Generator (default_rng), Docstrings, Nested loops, Lambda functions, pandas DataFrame and iloc slicing, Generator expressions with join

In [ ]:
# Tabular Q-learning for a single-SKU inventory reorder policy (stock 0..20, order 0/5/10)
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---- Environment settings (one SKU in one warehouse) ----
MAX_STOCK = 20
ACTIONS = [0, 5, 10]                 # units we can order each day
HOLDING_COST_PER_UNIT = 0.5          # $ per unit left on the shelf overnight
STOCKOUT_COST_PER_UNIT = 8.0         # $ per unit of demand we could not serve (lost margin + goodwill)
ORDER_FIXED_COST = 6.0               # $ per order placed (truck, paperwork)
MEAN_DAILY_DEMAND = 4.0


def step(stock, order_qty):
    """One day: receive order, see random demand, pay costs. Returns next_stock, reward, demand."""
    # 🧠 ML: demand is random (Poisson), so the agent must learn a policy that works on average
    demand = rng.poisson(MEAN_DAILY_DEMAND)
    # 🐍 PY: min() caps stock at shelf capacity; extra units are simply not accepted
    available = min(stock + order_qty, MAX_STOCK)
    sold = min(available, demand)
    next_stock = available - sold
    lost = demand - sold
    cost = HOLDING_COST_PER_UNIT * next_stock + STOCKOUT_COST_PER_UNIT * lost
    cost += ORDER_FIXED_COST if order_qty > 0 else 0.0
    # 🧠 ML: reward = -cost, so the agent maximising reward is minimising cost
    return next_stock, -cost, demand


# 🧠 ML: Q-table has one row per stock level (0..20) and one column per action
Q = np.zeros((MAX_STOCK + 1, len(ACTIONS)))
alpha, gamma = 0.1, 0.95
epsilon = 1.0
EPISODES, DAYS = 1500, 30

episode_costs = []
for ep in range(EPISODES):
    # 🐍 PY: rng.integers(a, b) returns a random int in [a, b) - random starting stock each episode
    stock = int(rng.integers(0, MAX_STOCK + 1))
    total_cost = 0.0
    for _ in range(DAYS):
        # 🧠 ML: epsilon-greedy - explore with probability epsilon, otherwise exploit best Q
        if rng.random() < epsilon:
            a = int(rng.integers(len(ACTIONS)))
        else:
            a = int(np.argmax(Q[stock]))
        nxt, reward, _ = step(stock, ACTIONS[a])
        # 🧠 ML: Bellman update - move Q toward (reward + discounted best value of next state)
        Q[stock, a] += alpha * (reward + gamma * Q[nxt].max() - Q[stock, a])
        total_cost -= reward
        stock = nxt
    episode_costs.append(total_cost)
    # 🧠 ML: decay exploration so the agent gradually trusts what it has learned
    epsilon = max(0.05, epsilon * 0.997)

# 🐍 PY: slicing [:100] and [-100:] takes first and last 100 episodes
print(f"Avg 30-day cost, first 100 episodes: ${np.mean(episode_costs[:100]):.1f}")
print(f"Avg 30-day cost, last 100 episodes : ${np.mean(episode_costs[-100:]):.1f}")

# 🐍 PY: build a pandas DataFrame so the learned policy prints as a neat table
policy = pd.DataFrame({
    "stock": np.arange(MAX_STOCK + 1),
    "best_order": [ACTIONS[int(np.argmax(Q[s]))] for s in range(MAX_STOCK + 1)],
    "Q_order0": Q[:, 0].round(1),
    "Q_order5": Q[:, 1].round(1),
    "Q_order10": Q[:, 2].round(1),
})
print("\nLearned policy (every 2nd stock level):")
print(policy.iloc[::2].to_string(index=False))


def evaluate(policy_fn, runs=300):
    """Run the greedy policy with no exploration and return mean 30-day cost."""
    costs = []
    for _ in range(runs):
        stock, c = 10, 0.0
        for _ in range(DAYS):
            stock, r, _ = step(stock, policy_fn(stock))
            c -= r
        costs.append(c)
    return np.mean(costs)


# 🐍 PY: lambda creates a tiny one-line function
q_cost = evaluate(lambda s: ACTIONS[int(np.argmax(Q[s]))])
never_cost = evaluate(lambda s: 0)
always_cost = evaluate(lambda s: 5)
print("\nPolicy evaluation (mean cost over 30 days, 300 runs):")
print(f"  Q-learning policy : ${q_cost:7.1f}")
print(f"  Always order 5    : ${always_cost:7.1f}")
print(f"  Never order       : ${never_cost:7.1f}")
# 🐍 PY: " ".join over a generator turns the 21 best actions into one compact line
print("\nFull policy, stock 0..20:", " ".join(str(x) for x in policy.best_order))
first_no_order = int(policy.loc[policy.best_order == 0, "stock"].min())
print(f"Business read-out: the agent mostly reorders when stock < {first_no_order} units,"
      f" saving ${always_cost - q_cost:.0f} per month vs a fixed 'order 5 daily' rule.")

**Try this next:**
- Change MEAN_DAILY_DEMAND to 6 and see whether the reorder point moves up.
- Set ORDER_FIXED_COST to 0: does the agent start ordering small amounts more often?
- Use a faster decay (0.99) and compare the last-100-episode cost.

## HARD: Lead time, tuning and an (s,S) showdown
Add a 2-day supplier lead time and weekly demand peaks, tune Q-learning, and test it against a tuned (s,S) policy on the same demand.

| 5W + H | |
|---|---|
| **Who** | A supply chain data science team deciding whether RL is worth deploying over the classic (s,S) rule. |
| **What** | A richer inventory environment with in-transit orders, weekend demand peaks, unit and fixed order costs, and a rigorous head-to-head test. |
| **When** | Before a pilot: policies are tuned on validation demand and judged on 400 unseen 8-week demand paths. |
| **Where** | A Python simulator that mirrors the warehouse's reorder process; outputs a saved Q-table for a pilot. |
| **Why** | An RL policy must beat or match a well-tuned textbook policy before anyone trusts it with real inventory dollars. |
| **How** | Inventory-position state, per-state decaying learning rate, grid search over alpha and epsilon decay, grid search over (s,S), common random numbers for a fair test. |

**🧠 ML concepts:** Markov state design (inventory position includes in-transit stock), Per-state-action decaying learning rate, Hyperparameter search for RL, Validation vs test separation for policies, Common random numbers for fair comparison, Baseline policy (s,S) from operations research  
**🐍 Python concepts:** Classes with __init__ and methods, Tuples as dict keys and setdefault, Closures (functions returning functions), itertools.product grid search, list.pop(0) as a simple queue, if __name__ == '__main__' guard

In [ ]:
# Production-style Q-learning for SKU reorder policy with a 2-day supplier lead time,
# hyperparameter tuning, and evaluation against a tuned (s, S) baseline on common random demand.
import os
import tempfile
import itertools
import numpy as np


class InventoryEnv:
    """Single SKU, daily review, orders arrive LEAD_TIME days after being placed."""
    ACTIONS = (0, 5, 10)

    def __init__(self, max_stock=25, lead_time=2, hold=0.5, stockout=8.0, order_fixed=6.0,
                 unit_cost=1.0, seed=0):
        self.max_stock, self.lead_time = max_stock, lead_time
        self.hold, self.stockout = hold, stockout
        self.order_fixed, self.unit_cost = order_fixed, unit_cost
        self.rng = np.random.default_rng(seed)

    def reset(self, stock=None):
        # 🐍 PY: "x if cond else y" is Python's one-line conditional expression
        self.on_hand = int(self.rng.integers(0, self.max_stock + 1)) if stock is None else stock
        # 🧠 ML: the pipeline (orders in transit) is part of the state - without it the problem is not Markov
        self.pipeline = [0] * self.lead_time
        return self.state()

    def state(self):
        # 🐍 PY: tuple(...) is hashable, so it can be used as a dictionary key for the Q-table
        return (self.on_hand,) + tuple(self.pipeline)

    def demand(self, day):
        # 🧠 ML: weekly seasonality (busier Fri/Sat) makes the problem harder than a fixed-mean demand
        mean = 4.0 + 2.0 * (day % 7 in (4, 5))
        return self.rng.poisson(mean)

    def step(self, action_idx, day, demand=None):
        qty = self.ACTIONS[action_idx]
        # 🐍 PY: list.pop(0) removes the oldest in-transit order: it arrives today
        arriving = self.pipeline.pop(0)
        self.pipeline.append(qty)
        self.on_hand = min(self.on_hand + arriving, self.max_stock)
        d = self.demand(day) if demand is None else demand
        sold = min(self.on_hand, d)
        lost = d - sold
        self.on_hand -= sold
        cost = (self.hold * self.on_hand + self.stockout * lost
                + (self.order_fixed + self.unit_cost * qty if qty else 0.0))
        return self.state(), -cost, cost


def full_state(env, day):
    # 🧠 ML: state = inventory position (on hand + in transit), capped at 40 -> only 41 states
    # 🧠 ML: we tested adding day-of-week too: 7x more states, slower to learn, worse with this budget
    position = env.on_hand + sum(env.pipeline)
    return (min(position, 40),)


def train_q(alpha0, eps_decay, episodes=3000, days=56, gamma=0.9, seed=1):
    env = InventoryEnv(seed=seed)
    # 🐍 PY: a dict keyed by state tuples; setdefault creates a zero row the first time a state is seen
    Q, visits = {}, {}
    eps = 1.0
    for ep in range(episodes):
        env.reset()
        s = full_state(env, 0)
        for day in range(days):
            q_s = Q.setdefault(s, np.zeros(3))
            n_s = visits.setdefault(s, np.zeros(3))
            a = int(env.rng.integers(3)) if env.rng.random() < eps else int(np.argmax(q_s))
            _, r, _ = env.step(a, day)
            s2 = full_state(env, day + 1)
            # 🧠 ML: per-(state,action) decaying learning rate alpha0 / n^0.6 - big steps early, small later
            n_s[a] += 1
            alpha = max(0.01, alpha0 / n_s[a] ** 0.6)
            # 🧠 ML: off-policy TD target uses the greedy max even when we explored
            q_s[a] += alpha * (r + gamma * Q.get(s2, np.zeros(3)).max() - q_s[a])
            s = s2
        # 🧠 ML: exponential epsilon decay with a floor, so exploration never fully stops in training
        eps = max(0.02, eps * eps_decay)
    return Q


def evaluate(policy_fn, demand_paths, start_stock=12):
    """Replay the SAME demand paths for every policy (common random numbers = fair comparison)."""
    env = InventoryEnv(seed=0)
    totals = []
    for path in demand_paths:
        env.reset(stock=start_stock)
        cost = 0.0
        for day, d in enumerate(path):
            a = policy_fn(env, day)
            _, _, c = env.step(a, day, demand=d)
            cost += c
        totals.append(cost)
    return np.mean(totals), np.std(totals)


def q_policy(Q):
    # 🐍 PY: a closure - the inner function "remembers" Q from the outer function
    def act(env, day):
        q = Q.get(full_state(env, day))
        return 0 if q is None else int(np.argmax(q))
    return act


def ss_policy(s_low, S_high):
    """Classic (s, S): if inventory position <= s, order up toward S (rounded to allowed lot sizes)."""
    def act(env, day):
        position = env.on_hand + sum(env.pipeline)
        if position > s_low:
            return 0
        gap = S_high - position
        # 🧠 ML: pick the allowed lot size closest to the gap - the baseline must respect the same actions
        return int(np.argmin([abs(gap - q) for q in InventoryEnv.ACTIONS]))
    return act


if __name__ == "__main__":
    rng = np.random.default_rng(42)
    days = 56
    # 🧠 ML: separate demand paths for tuning vs final test - avoid tuning on the test set
    tune_paths = [[rng.poisson(4.0 + 2.0 * (d % 7 in (4, 5))) for d in range(days)] for _ in range(150)]
    test_paths = [[rng.poisson(4.0 + 2.0 * (d % 7 in (4, 5))) for d in range(days)] for _ in range(400)]

    print("== Tuning Q-learning (alpha0 x epsilon decay) on validation demand ==")
    results = {}
    # 🐍 PY: itertools.product builds every combination of the two lists (a small grid search)
    for alpha0, decay in itertools.product([0.3, 0.6], [0.997, 0.999]):
        Q = train_q(alpha0, decay)
        mean_cost, _ = evaluate(q_policy(Q), tune_paths)
        results[(alpha0, decay)] = (mean_cost, Q)
        print(f"  alpha0={alpha0:<4} eps_decay={decay:<6} states_seen={len(Q):4d}  val_cost=${mean_cost:7.1f}")
    # 🐍 PY: min with key= picks the dict key whose stored cost is smallest
    best_cfg = min(results, key=lambda k: results[k][0])
    best_Q = results[best_cfg][1]
    print(f"  -> best config alpha0={best_cfg[0]}, eps_decay={best_cfg[1]}")

    print("\n== Tuning (s, S) baseline on the same validation demand ==")
    ss_grid = [(s, S) for s in range(6, 22, 2) for S in range(s + 5, 31, 3)]
    ss_scores = {cfg: evaluate(ss_policy(*cfg), tune_paths)[0] for cfg in ss_grid}
    best_ss = min(ss_scores, key=ss_scores.get)
    print(f"  searched {len(ss_grid)} (s,S) pairs -> best s={best_ss[0]}, S={best_ss[1]}"
          f" val_cost=${ss_scores[best_ss]:.1f}")

    print("\n== Final test on 400 unseen 8-week demand paths ==")
    rows = [("Q-learning (tuned)", q_policy(best_Q)),
            (f"(s,S)=({best_ss[0]},{best_ss[1]})", ss_policy(*best_ss)),
            ("Always order 5", lambda env, day: 1)]
    test_costs = {}
    for name, fn in rows:
        m, sd = evaluate(fn, test_paths)
        test_costs[name] = m
        bar = "#" * int(m / 40)
        print(f"  {name:20s} mean=${m:7.1f}  sd=${sd:5.1f}  {bar}")
    q_cost = test_costs["Q-learning (tuned)"]
    ss_cost = test_costs[f"(s,S)=({best_ss[0]},{best_ss[1]})"]
    print(f"\nQ-learning vs (s,S): {100 * (ss_cost - q_cost) / ss_cost:+.1f}% cost saving per 8 weeks")

    # Explainability: a tabular policy is fully readable - compare it to the (s,S) rule directly
    print("\nLearned order qty by inventory position (on hand + in transit):")
    # 🐍 PY: a list comprehension builds the row; a missing state means never visited -> show '-'
    acts = ["-" if best_Q.get((p,)) is None else str(InventoryEnv.ACTIONS[int(np.argmax(best_Q[(p,)]))])
            for p in range(0, 31)]
    print("  position: " + " ".join(f"{p:>2}" for p in range(0, 31, 1)))
    print("  order   : " + " ".join(f"{a:>2}" for a in acts))

    # 🧠 ML: model-risk note - an RL policy is a model; it needs validation, monitoring and a fallback
    print("\nModel-risk note: keep (s,S) as a fallback, cap order size, monitor service level weekly,"
          "\nand retrain when the demand pattern drifts (new promo calendar, new supplier lead time).")

    out_dir = tempfile.mkdtemp()
    path = os.path.join(out_dir, "q_table.npy")
    # 🐍 PY: np.save with allow_pickle stores the dict; production would use a versioned model registry
    np.save(path, np.array([best_Q, best_cfg, best_ss], dtype=object), allow_pickle=True)
    print(f"Saved Q-table + configs -> {os.path.basename(path)} in a temp dir")

**Try this next:**
- Set lead_time=4 in InventoryEnv and see which policy degrades more.
- Put day-of-week back into full_state and train 10,000 episodes: can RL now beat (s,S)?
- Raise the stockout cost to 20 and compare the new reorder point with the (s,S) answer.